In [23]:
import hashlib
import json

from shapely.strtree import STRtree
from scipy.spatial import cKDTree
from functools import lru_cache
import pandas as pd
import numpy as np
import geopandas as gpd
import sys
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from datetime import date
import pycountry
import rasterio

def extract_tifs_to_dataframe(df, raster_folder):
    """
    Extrai o valor de todos os TIFFs de uma pasta para um DataFrame.

    O DataFrame deve conter as colunas:
        latitude
        longitude

    Cada TIFF vira uma nova coluna.

    Exemplo:

    latitude longitude bio1 bio12 ndvi
    """

    df = df.copy()

    coords = [
        (lon, lat)
        for lat, lon in zip(df["latitude"], df["longitude"])
    ]

    raster_folder = Path(raster_folder)

    for tif in sorted(raster_folder.glob("*.tif")):

        variable = tif.stem

        print(f"Lendo {variable}...")

        with rasterio.open(tif) as src:

            values = [
                value[0]
                for value in src.sample(coords)
            ]

        values = [
            np.nan if src.nodata is not None and v == src.nodata else v
            for v in values
        ]

        df[variable] = values

    return df

In [24]:

@lru_cache(maxsize=None)
def _load_world_shapefile():
    URL = "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"
    return gpd.read_file(URL)

def get_country_grid(country_name: str, resolution_deg: float = 0.05) -> gpd.GeoDataFrame:
    country_code = pycountry.countries.search_fuzzy(country_name)[0].alpha_3
    world = _load_world_shapefile() 

    country_code = pycountry.countries.search_fuzzy(country_name)
    country_code = country_code[0].alpha_3

    country = world[world["ISO_A3"] == country_code]
    if country.empty:
        raise ValueError(f"País '{country_name}' não encontrado.")
    
    shape = country.union_all()
    
    lon_min, lat_min, lon_max, lat_max = shape.bounds
    
    lats = np.arange(lat_min, lat_max, resolution_deg)
    lons = np.arange(lon_min, lon_max, resolution_deg)
    lons_grid, lats_grid = np.meshgrid(lons, lats)
    points = gpd.points_from_xy(lons_grid.ravel(), lats_grid.ravel())

    tree = STRtree(points)
    mask = tree.query(shape, predicate="contains")

    grid_gdf = gpd.GeoDataFrame(geometry=points[mask], crs="EPSG:4326")
    
    grid_gdf["latitude"] = grid_gdf.geometry.y
    grid_gdf["longitude"] = grid_gdf.geometry.x
    print(f"Grid gerado com {len(grid_gdf)} pontos.")
    return grid_gdf


In [25]:
df = get_country_grid("china")

Grid gerado com 381818 pontos.


In [26]:
teste = extract_tifs_to_dataframe(df, "./tifs")

Lendo Aspect...
Lendo DEM_float...
Lendo POC...
Lendo Slope...
Lendo wc2.1_30s_bio_15...
Lendo wc2.1_30s_bio_18...
Lendo wc2.1_30s_bio_19...
Lendo wc2.1_30s_bio_2...
Lendo wc2.1_30s_bio_3...
Lendo wc2.1_30s_bio_6...
Lendo wc2.1_30s_bio_7...
Lendo wc2.1_30s_bio_8...


In [27]:
teste.to_csv("env.csv")